# Random Forest 



## What Is Random Forest Classification?

<span style="color:#1f77b4">**Random Forest Classification**</span> is an <span style="color:#1f77b4">**ensemble**</span> learning method that fits many <span style="color:#1f77b4">**decision trees**</span> and combines their class votes.

It combines:
- <span style="color:#1f77b4">**Bagging (Bootstrap Aggregation)**</span>: Fit each tree on a random subset of the data
- <span style="color:#ff7f0e">**Random Feature Subsets**</span>: At each split, choose from a random subset of features

---

### Key Features

- <span style="color:#2ca02c">**Reduces overfitting**</span> of single decision trees
- Handles nonlinear relationships well
- Provides built-in <span style="color:#2ca02c">**feature importance**</span>

### Key Hyperparameters

| Parameter | Meaning |
|-----------|---------|
| `n_estimators` | Number of trees in the forest |
| `max_depth` | Maximum depth of each tree |
| `max_features` | Number of features considered at each split |



Now that we have discussed Bagging, we can write down the following equations:

Random Forest = <span style="color:#1f77b4">**Bagging**</span> + <span style="color:#ff7f0e">**Random Subspace Method**</span>

At <span style="color:#ff7f0e">**every split**</span> inside each bootstrapped tree, choose only <span style="color:#ff7f0e">**$k$ of the $p$ features**</span> ($k < p$) as candidate split columns. This <span style="color:#2ca02c">**decorrelates trees**</span> even if a few predictors are overwhelmingly strong.

Why it Works

* Like bagging, averaging <span style="color:#2ca02c">**reduces variance**</span>.
* Extra <span style="color:#ff7f0e">**feature randomness**</span> prevents the “same strong feature first” syndrome, yielding <span style="color:#2ca02c">**even lower correlation**</span> between trees.




### How It Works

Conceptual algorithm (random forest = bagging + one extra twist at every split):

1. <span style="color:#1f77b4">**Bootstrap**</span> the training set $B$ times → $B$ equally-sized samples <span style="color:#ff7f0e">**with replacement**</span> (exactly as in bagging).
2. Grow an **independent, deep** tree on each sample --- but at <span style="color:#ff7f0e">**each split**</span>, draw a <span style="color:#ff7f0e">**random subset of $k$ of the $p$ features**</span> (`max_features`, a.k.a. *mtry*) and search for the best split <span style="color:#ff7f0e">**only among those $k$**</span>. A fresh subset is drawn at *every* split, not once per tree.
3. **Aggregate** by <span style="color:#1f77b4">**majority vote**</span> for classification (mean for regression).
4. **Model selection** (or parameter tuning) using <span style="color:#ff7f0e">**out-of-bag (OOB)**</span> errors.

> *A random forest becomes reliable by making sure its trees do not think alike: each sees only part of the evidence, so their individual errors fade when the whole forest speaks.*  --- Words of wisdom by GPT-5.6 Pro (2026)

> *A random forest is a wise committee built from a strange rule: no tree may look at all the evidence at once. Forced to differ, the trees stop making the same mistake --- and the vote comes out wiser than any member.* --- Words of wisdom by Claude Fable 5 (2026)

In [3]:
# --- Synthetic data: three classes separated by two straight lines -------
# (Same data-generating process as the Bagging companion; the "truth" is
#  known, so we can see exactly how well the forest recovers it.)
import numpy as np

rng = np.random.default_rng(0)   # fixed seed: everyone gets the same data
# Examples with more data points:
sample_size = 200
X = rng.uniform(0.1, 0.9, size=(sample_size, 2))   # 200 points, 2 features

# Assign labels by region (no noise -- the boundaries ARE the truth):
y = np.zeros(sample_size, dtype=int)
mask1 = X[:, 0] + X[:, 1] > 1.1            # above the line x1 + x2 = 1.1  -> class 1
mask2 = (~mask1) & (X[:, 0] - X[:, 1] > 0.3)  # below it but right of x1 - x2 = 0.3 -> class 0
y[mask1] = 1
y[mask2] = 0
y[~(mask1 | mask2)] = 2                    # everything else -> class 2


In [4]:
# ----- imports -------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import sklearn                                   # version check
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

import ipywidgets as widgets
from ipywidgets import interact

label_cmap = ListedColormap(["#1f77b4", "#ff7f0e", "#2ca02c"])

# ----- helper: fit & plot a random forest -----------------------------------
# The four algorithm steps from "How It Works" all happen inside
# RandomForestClassifier:
#   Step 1 (bootstrap):    bootstrap sampling WITH replacement is the default;
#                          one resample per tree, n_estimators trees in total.
#   Step 2 (random splits): max_features sets k, the number of features drawn
#                          at EACH split (a fresh subset per split -- this is
#                          the twist that separates a forest from plain bagging).
#   Step 3 (aggregate):    .predict() returns the majority vote of all trees.
#   Step 4 (OOB):          oob_score=True scores each point with the trees
#                          that never saw it -- free validation.
def plot_rf(max_features=1.0, n_estimators=100, max_depth=0):
    depth = None if max_depth == 0 else int(max_depth)   # 0 on the slider = unlimited depth

    rf = RandomForestClassifier(
        n_estimators=int(n_estimators),   # B: how many trees (Step 1)
        max_features=max_features,        # k: features per SPLIT (Step 2)
        max_depth=depth,                  # deep trees = unstable base learners
        oob_score=True,                   # Step 4: out-of-bag validation
        random_state=42
    ).fit(X, y)                           # Steps 1-3 run here

    # Visualization: ask the fitted forest to classify every point on a fine
    # grid, so the colored regions show the forest's decision rule (Step 3's
    # majority vote, evaluated everywhere).
    x_min, x_max = X[:, 0].min() - 0.05, X[:, 0].max() + 0.05
    y_min, y_max = X[:, 1].min() - 0.05, X[:, 1].max() + 0.05
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 400),
        np.linspace(y_min, y_max, 400)
    )
    Z = rf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    # plot decision regions + data
    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, Z, alpha=0.25, cmap=label_cmap)
    plt.scatter(X[:, 0], X[:, 1],
                c=y, cmap=label_cmap,
                edgecolors="k", s=80)

    depth_label = "None" if depth is None else depth
    plt.title(f"Random Forest: max_features={max_features:.2f}, "
              f"n_estimators={n_estimators}, max_depth={depth_label}\n"
              f"OOB accuracy ≈ {rf.oob_score_:.3f}")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.tight_layout()
    plt.show()

# ----- interactive widgets --------------------------------------------------
# What each slider teaches:
#   max_features: the forest's own dial (Step 2) -- lower it and the trees
#                 decorrelate; at 1.0 the forest collapses back to plain bagging
#   n_estimators: more trees -> steadier vote (it should NOT overfit as B grows)
#   max_depth:    deeper base trees = more variance for the vote to remove
interact(
    plot_rf,
    max_features = widgets.FloatSlider(
        value=1.0, min=0.2, max=1.0, step=0.1,
        description="max_features"
    ),
    n_estimators = widgets.IntSlider(
        value=100, min=10, max=300, step=10,
        description="n_estimators"
    ),
    max_depth = widgets.IntSlider(
        value=0, min=0, max=6, step=1,
        description="max_depth (0=None)"
    )
);


interactive(children=(FloatSlider(value=1.0, description='max_features', max=1.0, min=0.2), IntSlider(value=10…

> Check out the (slightly overly complicated but fun) visualization by MLU:  https://mlu-explain.github.io/random-forest/